In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Install Library**

In [3]:
!pip install -q transformers datasets evaluate accelerate emoji seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 15.9 MB/s eta 0:00:00


# **Import Library**

In [4]:
import re
import emoji
import torch
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    cohen_kappa_score
)

# **Load Dataset**

In [5]:
df = pd.read_parquet('/content/drive/MyDrive/Project Mandiri/Sentiment Analysis DANA/data/ulasan_dana_cleaned.parquet')

# **Mapping Rating → Sentiment**

In [6]:
def map_label(rating):
    if rating <= 2:
        return 0  # negatif
    elif rating == 3:
        return 1  # netral
    else:
        return 2  # positif

df["label"] = df["rating"].apply(map_label)

# **FILTER KUALITAS DATA (REALISTIC FILTER)**

# Buang emoji-only

In [7]:
def is_emoji_only(text):
    text_no_emoji = emoji.replace_emoji(str(text), replace='')
    text_no_emoji = re.sub(r'[^a-zA-Z]', '', text_no_emoji)
    return len(text_no_emoji.strip()) == 0

df = df[~df["ulasan"].apply(is_emoji_only)]

# Buang laughter/noise-only

In [8]:
def is_noise(text):
    text = str(text).lower().strip()
    return bool(re.fullmatch(r"(wkwk+|wk+|haha+|hehe+|hihi+)", text))

df = df[~df["ulasan"].apply(is_noise)]

# Minimal mengandung huruf (tanpa hapus 1 kata bermakna)

In [9]:
df = df[df["ulasan"].str.contains(r"[a-zA-Z]", regex=True)]
df = df[df["ulasan"].str.len() > 2]

In [10]:
df.to_csv("text_preprocessing_data.csv", index=False)

In [11]:
from google.colab import files
files.download("text_preprocessing_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Cek Distribusi Kelas Berdasarkan Label Setelah Dibersihkan**

In [12]:
df["label"].value_counts()

,count
label,
2,491025
0,181869
1,33491


# **Optional: Balancing Berdasarkan Kelas Minimum**

**Berdasarkan Kelas Minimum**

In [13]:
min_class_size = df["label"].value_counts().min()

print("Minimum class size:", min_class_size)

df_balanced = df.groupby("label").apply(
    lambda x: x.sample(n=min_class_size, random_state=42)
).reset_index(drop=True)

Minimum class size: 33491


/tmp/ipykernel_1330/3171276082.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby("label").apply(


# **Train Test Split**

In [14]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df_balanced["ulasan"],
    df_balanced["label"],
    test_size=0.2,
    stratify=df_balanced["label"],
    random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    stratify=temp_labels,
    random_state=42
)

# **Tokenization**

In [15]:
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_function(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=128
    )

train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)
test_encodings = tokenize_function(test_texts)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# **Convert to Dataset**

In [16]:
train_dataset = Dataset.from_dict({
    **train_encodings,
    "labels": train_labels.tolist()
})

val_dataset = Dataset.from_dict({
    **val_encodings,
    "labels": val_labels.tolist()
})

test_dataset = Dataset.from_dict({
    **test_encodings,
    "labels": test_labels.tolist()
})

# **Load Model**

In [17]:
model = XLMRobertaForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=3
)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# **Training Arguments**

In [18]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# **Metric Function**

In [19]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "kappa": cohen_kappa_score(labels, predictions)
    }

# **Trainer**

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Kappa
1,0.741819,0.712588,0.685777,0.676510,0.528665
2,0.700532,0.701127,0.689957,0.686618,0.534936


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Kappa
1,0.741819,0.712588,0.685777,0.676510,0.528665
2,0.700532,0.701127,0.689957,0.686618,0.534936
3,0.651500,0.703719,0.691550,0.686591,0.537325


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=15072, training_loss=0.7120670145990489, metrics={'train_runtime': 6622.1302, 'train_samples_per_second': 36.413, 'train_steps_per_second': 2.276, 'total_flos': 1.5861397717605888e+16, 'train_loss': 0.7120670145990489, 'epoch': 3.0})

# **Evaluation on Test Set**

In [21]:
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = test_labels.tolist()

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Cohen Kappa:", cohen_kappa_score(y_true, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, digits=4))

Accuracy: 0.6926751592356688
Macro F1: 0.6896657446571903
Cohen Kappa: 0.5390083030055096

Classification Report:

              precision    recall  f1-score   support

           0     0.7861    0.6363    0.7033      3349
           1     0.5719    0.5703    0.5711      3349
           2     0.7303    0.8713    0.7946      3350

    accuracy                         0.6927     10048
   macro avg     0.6961    0.6927    0.6897     10048
weighted avg     0.6961    0.6927    0.6897     10048



# **Confusion Matrix Visualisasi**

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
            xticklabels=["Neg", "Netral", "Pos"],
            yticklabels=["Neg", "Netral", "Pos"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [23]:
import plotly.express as px
import plotly.graph_objects as go

target_names = [str(i) for i in np.unique(y_true)]

In [24]:
report = classification_report(y_true, y_pred, target_names=target_names, output_dict=True, digits=4)
report_df = pd.DataFrame(report).iloc[:-1, :len(target_names)].T

fig_report = px.imshow(
    report_df,
    text_auto='.4f',
    labels=dict(color="Score"),
    x=['Precision', 'Recall', 'F1-Score'],
    y=report_df.index,
    title="Classification Report (Per-Class Metrics)",
    color_continuous_scale='RdBu_r'
)
fig_report.show()

In [25]:
metrics_data = {
    'Metric': ['Accuracy', 'Macro F1', 'Cohen Kappa'],
    'Value': [
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro"),
        cohen_kappa_score(y_true, y_pred)
    ]
}
metrics_df = pd.DataFrame(metrics_data)

fig_metrics = px.bar(
    metrics_df,
    x='Metric',
    y='Value',
    text_auto='.4f',
    color='Metric',
    title="Overall Performance Summary",
    range_y=[0, 1]
)
fig_metrics.update_layout(showlegend=False)
fig_metrics.show()

# **Inference Model**

In [34]:
label_map = {
    0: "negatif",
    1: "netral",
    2: "positif"
}

In [35]:
def predict_single(text, dukungan=0):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    pred_idx = int(np.argmax(probs))
    pred_label = label_map[pred_idx]

    impact = probs[0] * np.log1p(dukungan)

    return {
        "prob_negatif": probs[0],
        "prob_netral": probs[1],
        "prob_positif": probs[2],
        "pred_label": pred_label,
        "impact_score": impact
    }

In [36]:
predict_single("apk jelek", dukungan=120)

{'prob_negatif': np.float32(0.979111),
 'prob_netral': np.float32(0.008885962),
 'prob_positif': np.float32(0.012003024),
 'pred_label': 'negatif',
 'impact_score': np.float64(4.6956113526472505)}

In [38]:
predict_single("lumayan", dukungan=10)

{'prob_negatif': np.float32(0.045896534),
 'prob_netral': np.float32(0.8351351),
 'prob_positif': np.float32(0.11896832),
 'pred_label': 'netral',
 'impact_score': np.float64(0.11005508162066727)}

In [40]:
predict_single("enak banget pakai ni apk, banyak promonya", dukungan=25)

{'prob_negatif': np.float32(0.010851321),
 'prob_netral': np.float32(0.06405942),
 'prob_positif': np.float32(0.9250892),
 'pred_label': 'positif',
 'impact_score': np.float64(0.035354650766028506)}

In [42]:
predict_single("apk anomali", dukungan=3)

{'prob_negatif': np.float32(0.97390467),
 'prob_netral': np.float32(0.0096310275),
 'prob_positif': np.float32(0.016464235),
 'pred_label': 'negatif',
 'impact_score': np.float64(1.3501185512978786)}